# 🧪 Checkpoint Evaluator: Recursive Search & Comprehensive Analysis

This notebook systematically scans the entire Google Drive for model checkpoints, evaluates them against the test set, and identifies the weight files that best match our target performance metrics:

| Metric | Target |
|--------|--------|
| **F1-Score** | **0.86** |
| **AUC-ROC** | **0.89** |
| **Accuracy** | **0.84** |
| **Precision** | **0.84** |
| **Recall** | **0.88** |
| **PHQ-8 MAE** | **2.15** |

**Key Features:**
- 🔍 **Recursive Search**: Finds all `.pt` / `.pth` files in `DAIC-WOZ_Datasets`
- 🧠 **Deep Auto-Config**: Inspects checkpoint weights to dynamically detect dimensions (`d_model`, backbones) and stripped `module.` prefixes.
- 🛡️ **Robust Data Loading**: Handles nested dictionaries (like `quality_inputs`) safely during GPU transfer.
- ⚡ **Optimized Loading**: Reuses model instances when configuration matches, speeding up evaluation loop.
- 📊 **Full Evaluation**: Computes all 6 key metrics for every checkpoint
- 📈 **Visualization**: Generates confusion matrices and 5 comparative graphs
- 🏆 **Auto-Archival**: Saves the best model to the `achieved/` folder (even if targets missed)

## Step 1: Data & Environment Setup

In [ ]:
from google.colab import drive
import os
import sys
import shutil

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Clone/Update Repo
if not os.path.exists('/content/phase2'):
    !git clone https://github.com/nithin12342/phase2.git /content/phase2
else:
    %cd /content/phase2
    !git fetch origin
    !git reset --hard origin/main
    %cd /content

# 3. Install Dependencies
!pip install torch torchvision torchaudio transformers h5py pandas scikit-learn tqdm matplotlib seaborn --quiet

# 4. Add Project to Path
sys.path.insert(0, "/content/phase2/ml_pipeline/h5_omnifusion")
print("✅ Environment Ready")

## Step 2: Configuration & Target Metrics

In [ ]:
import torch
from pathlib import Path

# Paths
DATA_ROOT = "/content/drive/MyDrive/DAIC-WOZ_Datasets"
DATA_DIR = f"{DATA_ROOT}/H5_OmniFusion_Output"
LABELS_CSV = f"{DATA_DIR}/all_labels.csv"
ACHIEVED_DIR = f"{DATA_ROOT}/achieved"

# Configuration
BATCH_SIZE = 32
MAX_SEQ_LEN = 256
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Target Metrics
TARGETS = {
    "f1": 0.86,
    "auc": 0.89,
    "accuracy": 0.84,
    "precision": 0.84,
    "recall": 0.88,
    "phq8_mae": 2.15
}

# Ensure output directory exists
os.makedirs(ACHIEVED_DIR, exist_ok=True)
print(f"🎯 Targets set. Saving best models to: {ACHIEVED_DIR}")

## Step 3: Recursive Checkpoint Discovery

In [ ]:
import glob
import pandas as pd
import time

print("🔍 Scanning for checkpoints... (this may take a moment)")

checkpoint_files = []

# Recursive search for .pt and .pth files
search_patterns = [
    f"{DATA_ROOT}/**/*.pt",
    f"{DATA_ROOT}/**/*.pth"
]

for pattern in search_patterns:
    found = glob.glob(pattern, recursive=True)
    checkpoint_files.extend(found)

checkpoint_files = sorted(list(set(checkpoint_files)))

# Create a manifest
ckpt_data = []
for cf in checkpoint_files:
    try:
        stat = os.stat(cf)
        size_mb = stat.st_size / (1024 * 1024)
        mod_time = time.ctime(stat.st_mtime)
        ckpt_data.append({
            "filename": os.path.basename(cf),
            "path": cf,
            "size_mb": size_mb,
            "modified": mod_time
        })
    except Exception as e:
        print(f"⚠️ Error accessing {cf}: {e}")

df_ckpt = pd.DataFrame(ckpt_data)
if not df_ckpt.empty:
    print(f"✅ Found {len(df_ckpt)} unique checkpoint files.")
    display(df_ckpt[['filename', 'size_mb', 'modified']].head(10))
else:
    print("❌ No checkpoints found in Dicea/DAIC-WOZ folders!")

## Step 4: Data Loading
We use a **fixed test set** (Fold 0) to ensure fair comparison across all checkpoints.

In [ ]:
from src.data.h5_dataset import create_h5_dataloaders_kfold

# Create Test DataLoader
print("📂 Loading Test Data (Fold 0)...")
try:
    _, _, test_loader = create_h5_dataloaders_kfold(
        h5_dir=DATA_DIR,
        labels_csv=LABELS_CSV,
        batch_size=BATCH_SIZE,
        fold_idx=0,      # Fixed fold for consistency
        n_folds=5,
        max_seq_len=MAX_SEQ_LEN
    )
    print(f"✅ Test set ready: {len(test_loader.dataset)} samples")
except Exception as e:
    print(f"❌ Error creating dataloaders: {e}")
    test_loader = None

## Step 5: Deep Introspection Loop
This loop inspects weights to automatically infer configuration (`d_model` etc.) and handle `module.` prefixes.

In [ ]:
from config.model_config import H5Config, ComputeTier
from src.models.h5_omnifusion import H5OmniFusion
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score, 
    roc_auc_score, mean_absolute_error, confusion_matrix
)
import numpy as np
import gc
from collections import OrderedDict

# ------------------------------------------------------------
# RECURSIVE GPU MOVER
# ------------------------------------------------------------
def to_device(data, device):
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, dict):
        return {k: to_device(v, device) for k, v in data.items()}
    elif isinstance(data, list):
        return [to_device(v, device) for v in data]
    elif isinstance(data, tuple):
        return tuple(to_device(v, device) for v in data)
    else:
        return data

# Metrics Helper
def evaluate_checkpoint(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    all_phq_preds, all_phq_labels = [], []
    
    with torch.no_grad():
        for batch in loader:
            # Filter out non-input keys for model
            input_keys = [k for k in batch.keys() if k not in ['label', 'labels', 'target', 'targets', 'participant_id']]
            inputs = {k: batch[k] for k in input_keys}
            
            # Move inputs to device safely (handling nested dicts like quality_inputs)
            inputs = to_device(inputs, device)
            
            # Handle labels
            if 'label' in batch:
                labels = batch['label']['binary'].to(device)
                phq = batch['label']['phq_score'].to(device)
            elif 'targets' in batch:
                labels = batch['targets']['binary'].to(device)
                phq = batch['targets']['phq_score'].to(device)
            else:
                continue

            outputs = model(inputs)
            probs = outputs[0]['binary_prob']
            preds = (probs >= 0.5).long()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            
            if 'phq_score' in outputs[0]:
                all_phq_preds.extend(outputs[0]['phq_score'].cpu().numpy())
                all_phq_labels.extend(phq.cpu().numpy())

    y_true, y_pred, y_prob = np.array(all_labels), np.array(all_preds), np.array(all_probs)
    metrics = {}
    metrics['f1'] = f1_score(y_true, y_pred, zero_division=0)
    metrics['accuracy'] = accuracy_score(y_true, y_pred)
    metrics['precision'] = precision_score(y_true, y_pred, zero_division=0)
    metrics['recall'] = recall_score(y_true, y_pred, zero_division=0)
    try: metrics['auc'] = roc_auc_score(y_true, y_prob)
    except: metrics['auc'] = 0.5
    metrics['phq_mae'] = mean_absolute_error(all_phq_labels, all_phq_preds) if all_phq_labels else 99.9
    cm = confusion_matrix(y_true, y_pred)
    return metrics, cm

# ------------------------------------------------------------
# DEEP INTROSPECTION LOADER
# ------------------------------------------------------------
current_model = None

def load_introspected_model(state_dict, filename):
    global current_model
    
    # 1. Start with best guess based on filename
    filename = filename.lower()
    if "nano" in filename: tier = "nano"
    elif "micro" in filename: tier = "micro"
    else: tier = "medium"
    
    config = H5Config.from_tier(ComputeTier(tier))
    
    # 2. Inspect state_dict for TRUE dimensions
    def find_weight(name_suffix):
        for k, v in state_dict.items():
            if k.endswith(name_suffix):
                return v
        return None

    # Detect d_model from audio projection: [d_model, input_dim]
    audio_w = find_weight('audio_encoder.input_proj.weight')
    if audio_w is not None:
        detected_d_model = audio_w.shape[0]
        detected_audio_dim = audio_w.shape[1]
        
        if detected_d_model != config.d_model:
            print(f"⚠️ Auto-Fix: Detected d_model={detected_d_model} (expected {config.d_model}). Updating.")
            config.d_model = detected_d_model
            
        if detected_audio_dim != config.audio.backbone_dim:
            print(f"⚠️ Auto-Fix: Detected audio_dim={detected_audio_dim} (expected {config.audio.backbone_dim}). Updating.")
            config.audio.backbone_dim = detected_audio_dim
            
    # Detect other modality dims
    text_w = find_weight('text_encoder.input_proj.weight')
    if text_w is not None and text_w.shape[1] != config.text.backbone_dim:
         config.text.backbone_dim = text_w.shape[1]
         print(f"⚠️ Auto-Fix: Detected text_dim={config.text.backbone_dim}")

    video_w = find_weight('video_encoder.input_proj.weight')
    if video_w is not None and video_w.shape[1] != config.video.backbone_dim:
         config.video.backbone_dim = video_w.shape[1]
         print(f"⚠️ Auto-Fix: Detected video_dim={config.video.backbone_dim}")

    # 3. CHECK IF WE CAN REUSE CURRENT MODEL
    # Check if we have a loaded model and if its config matches
    if current_model is not None:
        curr_c = current_model.config
        # Compare critical dimensions
        if (curr_c.d_model == config.d_model and
            curr_c.audio.backbone_dim == config.audio.backbone_dim and
            curr_c.text.backbone_dim == config.text.backbone_dim and
            curr_c.video.backbone_dim == config.video.backbone_dim):
            
            print(f"🔄 Reuse: Model configuration matches (d_model={config.d_model}). Skipping re-init.")
            return current_model

    # 4. If not reusable, Re-instantiate
    if current_model is not None:
        del current_model
        torch.cuda.empty_cache()
        gc.collect()
    
    print(f"🔧 Instantiating model (Tier: {tier.upper()}, d_model: {config.d_model})...")
    current_model = H5OmniFusion(config)
    current_model.to(DEVICE)
    
    return current_model


# Main Loop
results_table = []
confusion_matrices = {}

print(f"🚀 Starting evaluation of {len(df_ckpt)} checkpoints...")

for idx, row in df_ckpt.iterrows():
    ckpt_path = row['path']
    ckpt_name = row['filename']
    
    print(f"\n[{idx+1}/{len(df_ckpt)}] Evaluating {ckpt_name}...")
    
    try:
        # 1. Load weights safely (CPU first)
        checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
        
        # 2. Extract state_dict
        state_dict = None
        if isinstance(checkpoint, dict):
            if 'model_state_dict' in checkpoint: state_dict = checkpoint['model_state_dict']
            elif 'state_dict' in checkpoint: state_dict = checkpoint['state_dict']
            else: state_dict = checkpoint
        elif hasattr(checkpoint, 'state_dict'):
            state_dict = checkpoint.state_dict()
        else:
            print(f"❌ Unknown checkpoint format for {ckpt_name}")
            continue

        # 3. Handle 'module.' prefix (DataParallel)
        new_state_dict = OrderedDict()
        for k, v in state_dict.items():
            name = k[7:] if k.startswith('module.') else k 
            new_state_dict[name] = v
        state_dict = new_state_dict
            
        # 4. Create Model compatible with these weights (OPTIMIZED REUSE)
        model = load_introspected_model(state_dict, ckpt_name)
        
        # 5. Load weights
        msg = model.load_state_dict(state_dict, strict=False)
        if msg.missing_keys:
             print(f'    ⚠️ Missing keys: {len(msg.missing_keys)} (usually fine for fine-tuning)')
        if msg.unexpected_keys:
             print(f'    ⚠️ Unexpected keys: {len(msg.unexpected_keys)}')

        # 6. Evaluate
        metrics, cm = evaluate_checkpoint(model, test_loader, DEVICE)
        
        conf = model.config
        result_entry = {
            'Checkpoint': ckpt_name,
            'Path': ckpt_path,
            'Config': f"{conf.d_model}dim",
            'F1': metrics['f1'],
            'AUC': metrics['auc'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'PHQ_MAE': metrics['phq_mae']
        }
        results_table.append(result_entry)
        confusion_matrices[ckpt_name] = cm
        
        print(f"   F1: {metrics['f1']:.4f} | AUC: {metrics['auc']:.4f} | Acc: {metrics['accuracy']:.4f} | MAE: {metrics['phq_mae']:.4f}")
        
    except Exception as e:
        print(f"❌ Failed to load/evaluate {ckpt_name}: {str(e)[:200]}...")

## Step 6: Analysis & Best Model Selection

In [ ]:
model_results = pd.DataFrame(results_table)

if not model_results.empty:
    def calculate_score(row):
        score = 0
        score += (row['F1'] - TARGETS['f1']) * 2.0
        score += (row['AUC'] - TARGETS['auc'])
        score += (row['Accuracy'] - TARGETS['accuracy'])
        score += (TARGETS['phq8_mae'] - row['PHQ_MAE']) * 0.5
        return score

    model_results['Score'] = model_results.apply(calculate_score, axis=1)
    model_results = model_results.sort_values(by='Score', ascending=False).reset_index(drop=True)
    
    print("🏆 Top 5 Checkpoints:")
    display(model_results[['Checkpoint', 'Config', 'F1', 'AUC', 'Accuracy', 'Recall', 'Precision', 'PHQ_MAE', 'Score']].head(5))
    
    best_model_row = model_results.iloc[0]
    best_ckpt_path = best_model_row['Path']
    print(f"\n🌟 BEST MODEL: {best_model_row['Checkpoint']}")
else:
    print("❌ No results to analyze.")

## Step 7: Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not model_results.empty:
    metrics_to_plot = ['F1', 'AUC', 'Accuracy', 'Precision', 'Recall']
    fig, axes = plt.subplots(1, 5, figsize=(25, 5))
    
    top_n = model_results.head(10)

    for i, metric in enumerate(metrics_to_plot):
        ax = axes[i]
        sns.barplot(data=top_n, x='Checkpoint', y=metric, ax=ax, palette='viridis')
        ax.set_title(f'{metric} (Target: {TARGETS[metric.lower()]})')
        ax.axhline(TARGETS[metric.lower()], color='r', linestyle='--', label='Target')
        ax.tick_params(axis='x', rotation=90)
        ax.set_ylim(0, 1.05)
        ax.legend()
    
    plt.tight_layout()
    plt.show()

    best_cm = confusion_matrices[best_model_row['Checkpoint']]
    plt.figure(figsize=(6, 5))
    sns.heatmap(best_cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Non-Depressed', 'Depressed'],
                yticklabels=['Non-Depressed', 'Depressed'])
    plt.title(f"Confusion Matrix: {best_model_row['Checkpoint']}")
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.show()

## Step 8: Archive Best Model (Auto-Fallback)

In [ ]:
import json
import time
import shutil

if not model_results.empty:
    print(f"\n📊 Analysis Complete.")
    print(f"   Best Available Model: {best_model_row['Checkpoint']}")
    print(f"   Source Path: {best_ckpt_path}")
    
    targets_met = True
    missed_metrics = []
    
    if best_model_row['F1'] < TARGETS['f1']: 
        targets_met = False
        missed_metrics.append(f"F1 ({best_model_row['F1']:.4f} < {TARGETS['f1']})")
    if best_model_row['AUC'] < TARGETS['auc']:
        targets_met = False
        missed_metrics.append(f"AUC ({best_model_row['AUC']:.4f} < {TARGETS['auc']})")
    if best_model_row['Accuracy'] < TARGETS['accuracy']:
        targets_met = False
        missed_metrics.append(f"Accuracy ({best_model_row['Accuracy']:.4f} < {TARGETS['accuracy']})")
        
    if targets_met:
        print(f"\n✅ SUCCESS! Target metrics achieved.")
        dest_folder = ACHIEVED_DIR
    else:
        print(f"\n⚠️ TARGETS NOT FULLY MET.")
        print(f"   Missed: {', '.join(missed_metrics)}")
        print(f"   Saving consistently to 'achieved' folder as the best candidate for further training.")
        dest_folder = ACHIEVED_DIR
    
    print(f"\n💾 Saving model to {dest_folder}...")
    target_path = os.path.join(dest_folder, f"BEST_{best_model_row['Checkpoint']}")
    
    shutil.copy2(best_ckpt_path, target_path)
    print(f"✅ Model copied: {target_path}")
    
    report = best_model_row.to_dict()
    report['timestamp'] = time.ctime()
    report['original_path'] = best_ckpt_path
    report['targets_met'] = targets_met
    report['missed_metrics'] = missed_metrics
    
    with open(os.path.join(dest_folder, "best_metrics_report.json"), 'w') as f:
        json.dump(report, f, indent=4)
    
    print("✅ Metrics report saved.")
    
    if not targets_met:
        print(f"\n👉 RECOMMENDED ACTION: Resume training from this checkpoint.")
        print(f"   Path: {target_path}")